In [0]:
%python

import mlflow
import mlflow.sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

mlflow.set_experiment("/Users/alliesdl@gmail.com/credit_risk_model")

df = (spark.table("lender_portfolio.gold.loan_table")
      .filter("matured")
      .toPandas())
print("matured loans loaded:", len(df))

In [0]:
%python
NUMERIC = ["principal", "term_months", "interest_rate_pct", "monthly_income",
           "employment_months", "bureau_score", "n_prior_applications",
           "age_at_application", "dti", "loan_to_income", "bureau_no_hit"]
CATEGORICAL = ["province", "employer_type", "channel_type"]
TARGET = "default_flag"

df["bureau_no_hit"] = df["bureau_no_hit"].astype(float)
df[CATEGORICAL] = df[CATEGORICAL].fillna("Unknown")

df = df.sort_values("application_date")
cut = int(len(df) * 0.8)
train, test = df.iloc[:cut], df.iloc[cut:]

X_train, y_train = train[NUMERIC + CATEGORICAL], train[TARGET].astype(int)
X_test, y_test = test[NUMERIC + CATEGORICAL], test[TARGET].astype(int)

print("train:", len(train), "loans up to", train["application_date"].max())
print("test: ", len(test), "loans from", test["application_date"].min())
print("default rate train / test:", round(y_train.mean(), 3), "/", round(y_test.mean(), 3))

In [0]:
%python
numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
categorical_pipe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

prep = ColumnTransformer([
    ("num", numeric_pipe, NUMERIC),
    ("cat", categorical_pipe, CATEGORICAL),
])

In [0]:
%python
models = {
    "logistic_regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "gradient_boosting": HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,
                                                        random_state=42),
}

for name, clf in models.items():
    with mlflow.start_run(run_name=name):
        pipe = Pipeline([("prep", clone(prep)), ("model", clf)])
        pipe.fit(X_train, y_train)

        p_default = pipe.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, p_default)
        gini = 2 * auc - 1

        mlflow.log_params({"model": name, "n_train": len(train), "n_test": len(test),
                           "split": "time-based 80/20", "features": len(NUMERIC + CATEGORICAL)})
        mlflow.log_metrics({"test_auc": auc, "test_gini": gini})
        mlflow.sklearn.log_model(pipe, "model")

        print(f"{name}: AUC = {auc:.3f} | Gini = {gini:.3f}")